# Novel Continual Learning (v7) in JupyterLab on KENET Chui

This notebook runs the thesis experiment (Static, Naive FT, Periodic, Novel, DoRA, LoRA+, O-LoRA on the synthetic and Sheng streams, over several seeds) from **JupyterLab on Chui's Open OnDemand portal**. It uses only folders on the cluster: nothing here depends on Google Drive or Colab.

**Before you start**
1. Connect the KENET VPN, open **https://ondemand.vlab.ac.ke**, and start **Interactive Apps → JupyterLab**. In the form choose the GPU partition **`gpu1`**, **1 GPU**, 4 cores and as many hours as you need (24 at most). *TODO(Chui): the form's field names are not published; pick the GPU options it offers.*
2. When the session starts, click **Connect to Jupyter**. In the JupyterLab file browser (left), go to your home folder and upload (drag and drop, or the ↑ button):
   - `novel_continual_learning_v7.zip`
   - `sheng_clean.csv`
   - this notebook, `run_v7_chui.ipynb`
3. Open this notebook and run the cells from the top. Any kernel works (the default Python 3 is fine): the heavy work runs in the experiment's own Python environment, which cell 4 creates.

**Jupyter or batch jobs?** A JupyterLab session is itself a SLURM job on `gpu1`, so it stops at its time limit (24 h at most), and so does anything running inside it. The full experiment (70 runs) takes longer than that on one GPU. Two ways to handle it, both in section 7:
- **7a**: run in the background of this session, and after the session ends start a new one and run 7a again: `--resume` skips every run that already finished.
- **7b** (recommended for the full run): submit the batch job array from this notebook. It runs on its own, up to 2 GPUs at once, whether or not this session is open. Then end this session so it doesn't hold a GPU.

## 1. Settings
Paths are on the cluster. Keep them the same as in `slurm/chui_env.sh` so the notebook and the batch jobs share data, the model cache and finished runs. The cell looks for the zip and the CSV in your home folder, the storage folder and the folder this notebook is in, and copies the CSV into the storage folder.

In [ ]:
import os, sys, glob, json, shutil, signal, subprocess, time, zipfile, platform

HOME = os.path.expanduser("~")
CODE_DIR        = os.path.join(HOME, "novel_continual_learning_v7")      # the unpacked code
EXPERIMENT_ROOT = os.path.join(HOME, "localscratch", "thesis_v7")        # data, finished runs, results (TODO(Chui): confirm ~/localscratch is not purged)
ENV_DIR         = os.path.join(HOME, "envs", "ncl_v7")                   # Python env made by slurm/setup_env.sh
HF_HOME         = os.path.join(EXPERIMENT_ROOT, "hf_cache")              # Qwen2.5-1.5B download (~3 GB)

# What to run (sections 6-8)
SEEDS      = "42 43 44 45 46"
MODELS     = "all"               # or "core", "variants", or e.g. "novel,dora,olora"
DATASETS   = "synthetic,sheng"   # or "sheng"
EXTRA_ARGS = ""                  # e.g. "--ft-mode lora"; use the same value for runs and the summary

os.makedirs(EXPERIMENT_ROOT, exist_ok=True)
for k, v in dict(CODE_DIR=CODE_DIR, EXPERIMENT_ROOT=EXPERIMENT_ROOT, ENV_DIR=ENV_DIR, HF_HOME=HF_HOME).items():
    os.environ[k] = v              # every command below inherits these

SEARCH = list(dict.fromkeys([HOME, EXPERIMENT_ROOT, os.getcwd(), CODE_DIR,
                             os.path.join(HOME, "localscratch"), os.path.join(HOME, "Downloads")]))
def find(name):
    for d in SEARCH:
        p = os.path.join(d, name)
        if os.path.isfile(p):
            return p
    return None

ZIP = find("novel_continual_learning_v7.zip")
CSV = find("sheng_clean.csv")
SHENG_CSV = os.path.join(EXPERIMENT_ROOT, "sheng_clean.csv")
if CSV and os.path.abspath(CSV) != SHENG_CSV:
    shutil.copy2(CSV, SHENG_CSV)
os.environ["SHENG_CSV"] = SHENG_CSV

print(f"Code:            {CODE_DIR}")
print(f"Storage root:    {EXPERIMENT_ROOT}")
print(f"Python env:      {ENV_DIR}")
print(f"HF cache:        {HF_HOME}")
print(f"{'OK     ' if ZIP else 'MISSING'}  code zip   {ZIP or 'novel_continual_learning_v7.zip: upload it to ' + HOME}")
print(f"{'OK     ' if os.path.isfile(SHENG_CSV) else 'MISSING'}  Sheng CSV  "
      f"{SHENG_CSV if os.path.isfile(SHENG_CSV) else 'sheng_clean.csv: upload it to ' + HOME + ', then re-run this cell'}")

### Helper
`sh()` runs a shell command in the code folder and prints its output as it comes. `login=True` uses a login shell, which is what makes the `module` command available. `running_pid()` tells whether the background run of section 7a is alive.

In [ ]:
def sh(cmd, login=False, check=True, cwd=None):
    p = subprocess.Popen(["bash", "-lc" if login else "-c", cmd], cwd=cwd or CODE_DIR,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")
    return rc if rc else None   # nothing printed under the cell on success

# Background run (section 7a): its log, and a file recording which node and PID it runs as.
RUN_LOG = os.path.join(EXPERIMENT_ROOT, "jupyter_run.log")
PID_FILE = os.path.join(EXPERIMENT_ROOT, "jupyter_run.pid")

def running_pid():
    """PID of the section-7a run if it is still alive on this node, else None."""
    try:
        host, pid = open(PID_FILE).read().split(":")
        pid = int(pid)
    except (OSError, ValueError):
        return None
    if host != platform.node():
        return None          # started in an earlier session on another node; that session has ended
    p = globals().get("proc")
    if p is not None and p.pid == pid:
        return pid if p.poll() is None else None     # also reaps it once finished
    try:
        os.kill(pid, 0)
        if open(f"/proc/{pid}/stat").read().split()[2] == "Z":
            return None      # finished, not yet reaped
    except OSError:
        return None
    return pid

## 2. Where is this session running?
You should see a `gpu1` node and a GPU. If `nvidia-smi` fails, this session has no GPU: end it and start JupyterLab again with a GPU (setup in section 4 still works without one).

In [ ]:
print("host:", platform.node())
print("SLURM job:", os.environ.get("SLURM_JOB_ID", "none (not inside a SLURM job)"),
      "| partition:", os.environ.get("SLURM_JOB_PARTITION", "?"),
      "| time limit: see `squeue -j $SLURM_JOB_ID -o %l`")
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo 'no GPU visible'", check=False, cwd=HOME)
sh("command -v sbatch >/dev/null && echo 'sbatch available: section 7b will work' "
   "|| echo 'sbatch not found here: use section 7a, or submit from a login shell'", login=True, check=False, cwd=HOME)

## 3. Unpack the code
Unpacks the zip into `CODE_DIR`. Re-run it after uploading a newer zip; it overwrites the code but never touches the storage folder (data, finished runs, results).

In [ ]:
assert ZIP, "Upload novel_continual_learning_v7.zip first (section 1)."
os.makedirs(CODE_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall(CODE_DIR)
for f in glob.glob(os.path.join(CODE_DIR, "slurm", "*")):
    os.chmod(f, 0o755)
print(sorted(os.listdir(CODE_DIR)))

## 4. One-time setup: on the login node (about 15 minutes)
Setup downloads Python packages and the Qwen2.5-1.5B model, which needs internet. **Chui's compute nodes, including this JupyterLab session, have no internet**, so setup runs once on the **login node**:

1. In OnDemand, open **Clusters → Shell Access** (or `ssh -p PORT you@HOSTNAME` from your laptop).
2. Run:
   ```bash
   cd ~/novel_continual_learning_v7 && bash slurm/setup_env.sh
   ```
   (It removes any half-made env left by an earlier attempt from Jupyter.)
3. Come back here and run the cell below. It only checks that setup finished (and runs setup itself if this node happens to have internet).

Setup creates the env in `ENV_DIR` (Python 3.11 via the Chui conda module), installs `requirements.txt`, removes `torchao`, downloads the model into `HF_HOME`, exports the Sheng eras, generates the synthetic stream and writes the package versions to `environment.txt`. After that, nothing needs internet: this notebook and the batch jobs load everything from disk.

*If the login node has no internet either*, ask KENET support (support@kenet.or.ke) whether there is an HTTP proxy for pip and Hugging Face; with one, `export https_proxy=...` before running setup.

In [ ]:
import socket
def online(host):
    try:
        socket.getaddrinfo(host, 443)
        return True
    except OSError:
        return False

FORCE_SETUP = False          # True, or "skip-env"
done_marker = os.path.join(EXPERIMENT_ROOT, "environment.txt")
env_ready = os.path.exists(os.path.join(ENV_DIR, ".ncl_v7_ready")) or os.path.exists(done_marker)
if env_ready and os.path.exists(done_marker) and not FORCE_SETUP:
    print(f"✅ Setup is done ({done_marker}). Continue with section 5.")
elif online("pypi.org") and online("huggingface.co"):
    assert os.path.isfile(SHENG_CSV), f"{SHENG_CSV} is missing: upload sheng_clean.csv and re-run section 1."
    sh("bash slurm/setup_env.sh" + (" --skip-env" if FORCE_SETUP == "skip-env" else ""), login=True)
else:
    print(f"❌ Setup has not finished, and this node ({platform.node()}) has no internet.\n"
          "   Open OnDemand -> Clusters -> Shell Access and run:\n\n"
          f"     cd {CODE_DIR} && bash slurm/setup_env.sh\n\n"
          "   Then run this cell again.")

## 5. Use the experiment's environment, offline
Every command from here on runs with the env's Python and loads the model from the local cache (no Hugging Face access needed).

In [ ]:
PY = os.path.join(ENV_DIR, "bin", "python")
assert os.path.exists(PY), f"{PY} not found: run section 4 first."
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", HF_DATASETS_OFFLINE="1",
                  TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1")
sh(f'"{PY}" -c "import torch, transformers, peft; print(\'torch\', torch.__version__, \'| CUDA available:\', '
   f'torch.cuda.is_available(), \'| transformers\', transformers.__version__, \'| peft\', peft.__version__)"')

### (Only if you change `sheng_clean.csv`) Re-export the Sheng eras
The CSV path is passed explicitly, so the export never looks for a Drive or Windows path.

In [ ]:
assert os.path.isfile(SHENG_CSV), f"{SHENG_CSV} is missing"
sh(f'"{PY}" -m data.export_sheng --csv "{SHENG_CSV}" --storage-root "{EXPERIMENT_ROOT}"')

## 6. Quick check (about 10-30 minutes on a GPU)
5 synthetic eras × 200 and 50 Sheng posts per era, all seven models, seed 42. It uses its own subfolder (`quick_test`), so its small synthetic stream never replaces the full one. Run it once before the full experiment.

In [ ]:
QUICK_ROOT = os.path.join(EXPERIMENT_ROOT, "quick_test")
src = os.path.join(EXPERIMENT_ROOT, "novel_continual_learning", "data", "sheng")
dst = os.path.join(QUICK_ROOT, "novel_continual_learning", "data", "sheng")
shutil.copytree(src, dst, dirs_exist_ok=True)
sh(f'"{PY}" run_experiment.py --storage-root "{QUICK_ROOT}" --quick --resume {EXTRA_ARGS}')

## 7a. Full run inside this session (background)
Starts the run as a background process, so you can close the browser tab and come back; output goes to a log file in the storage folder. It stops when this JupyterLab session ends. **To continue, start a new session, run sections 1, 2 (optional), 5 and this cell again**: `--resume` skips every (seed, dataset, model) run already finished, so only the interrupted one starts over.

Don't run 7a and 7b on the same seeds at the same time.

In [ ]:
if running_pid():
    print(f"Already running (PID {running_pid()}). Watch it with the next cell.")
else:
    cmd = (f'"{PY}" run_experiment.py --seeds {SEEDS} --models {MODELS} --datasets {DATASETS} '
           f'--resume {EXTRA_ARGS}')
    log = open(RUN_LOG, "a")
    log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} on {platform.node()}: {cmd}\n"); log.flush()
    proc = subprocess.Popen(["bash", "-c", cmd], cwd=CODE_DIR, stdout=log, stderr=subprocess.STDOUT,
                            start_new_session=True)
    open(PID_FILE, "w").write(f"{platform.node()}:{proc.pid}")
    print(f"Started PID {proc.pid}: {cmd}\nLog: {RUN_LOG}")

### Watch progress
Re-run this cell whenever you like. It shows whether the run is alive, the last lines of the log, and how many runs have finished (from 7a and 7b alike).

In [ ]:
def finished_runs():
    files = glob.glob(os.path.join(EXPERIMENT_ROOT, "novel_continual_learning", "checkpoints", "v7_*",
                                   "seed*", "*", "*.json"))
    return sorted(os.path.relpath(f, os.path.join(EXPERIMENT_ROOT, "novel_continual_learning", "checkpoints"))
                  for f in files if "quick_test" not in f)

pid = running_pid()
print("Background run:", f"running (PID {pid})" if pid else "not running")
if os.path.exists(RUN_LOG):
    lines = open(RUN_LOG, errors="replace").read().replace("\r", "\n").splitlines()
    print("\n--- last log lines ---")
    print("\n".join(l for l in lines[-25:]))
runs = finished_runs()
print(f"\n{len(runs)} finished run(s) on disk" + (": latest " + runs[-1] if runs else ""))
sh("squeue -u $USER 2>/dev/null || true", login=True, check=False, cwd=HOME)

### Stop the background run
Finished runs are kept; the one in progress is lost.

In [ ]:
pid = running_pid()
if pid:
    os.killpg(os.getpgid(pid), signal.SIGTERM)
    print(f"Stopped PID {pid}")
else:
    print("Nothing running")

## 7b. Full run as batch jobs (recommended)
Submits the job array (one task per seed × model group, at most 2 GPUs at once) and a summary job that runs after it, exactly as `bash slurm/submit_all.sh` from a login shell. The jobs keep running after this session ends, so **end this JupyterLab session afterwards** (OnDemand → My Interactive Sessions → Delete) to free its GPU. Come back later in a short session (no GPU needed) to watch progress (cell above) and read the results (section 8).

`MODEL_GROUPS` sets how the array is split: `"core variants"` (10 tasks for 5 seeds), `"all"` (one per seed), or one per model if a task needs more than 24 h. *TODO(Chui): if the jobs are rejected, see README_v7 step 4 (`--mem`, QOS, account lines in `slurm/*.sbatch`).*

In [ ]:
MODEL_GROUPS = "core variants"
os.environ.update(SEEDS=SEEDS, MODEL_GROUPS=MODEL_GROUPS, DATASETS=DATASETS, EXTRA_ARGS=EXTRA_ARGS)
sh("bash slurm/submit_all.sh", login=True)

## 8. Results: mean ± std over seeds
Builds the summary from every finished run on disk (from 7a, 7b or both), for the seeds, models and datasets set in section 1, and shows it. Runs that have not finished yet are listed at the top of the output. Files are written to `novel_continual_learning/experiments/` in the storage folder (`aggregate_<id>_summary.md / .csv / .json`, and `aggregate_<id>.json` with every accuracy matrix).

In [ ]:
from IPython.display import Markdown, display
sh(f'"{PY}" run_experiment.py --aggregate-only --seeds {SEEDS} --models {MODELS} --datasets {DATASETS} '
   f'{EXTRA_ARGS} > "{EXPERIMENT_ROOT}/last_aggregate.txt" 2>&1; grep -E "finished run|not finished|Results:" '
   f'"{EXPERIMENT_ROOT}/last_aggregate.txt"', check=False)
EXP_DIR = os.path.join(EXPERIMENT_ROOT, "novel_continual_learning", "experiments")
latest = sorted(glob.glob(os.path.join(EXP_DIR, "aggregate_*_summary.md")), key=os.path.getmtime)
if latest:
    print("Showing", latest[-1])
    display(Markdown(open(latest[-1], encoding="utf-8").read()))

## 9. (Optional) Accuracy matrix of one run
`R[r, j]` = accuracy on era j's test set after training on r eras; forgetting shows up as values falling down a column. Drawn with the env's matplotlib, so it works whatever kernel this notebook uses.

In [ ]:
from IPython.display import Image
SEED, DATASET, MODEL = "42", "sheng", "Novel-Sheng"      # synthetic models have no "-Sheng" suffix
agg = sorted(glob.glob(os.path.join(EXP_DIR, "aggregate_*[0-9].json")), key=os.path.getmtime)
assert agg, "Run section 8 first."
have = json.load(open(agg[-1]))["results_by_seed"]
assert MODEL in have.get(SEED, {}).get(DATASET, {}), (
    f"No finished {MODEL} run for seed {SEED} on {DATASET}. Available: "
    + "; ".join(f"seed {s} {d}: {', '.join(m)}" for s, ds in have.items() for d, m in ds.items()))
png = os.path.join(EXPERIMENT_ROOT, f"acc_matrix_{MODEL}_{SEED}.png")
script = f"""
import json, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
res = json.load(open({agg[-1]!r}))
R = np.array(res["results_by_seed"][{SEED!r}][{DATASET!r}][{MODEL!r}]["acc_matrix"], dtype=float)
plt.figure(figsize=(7, 6)); plt.imshow(R, vmin=0, vmax=1, cmap="viridis", aspect="auto")
plt.colorbar(label="accuracy"); plt.xlabel("test era j"); plt.ylabel("eras trained (r)")
plt.title({MODEL!r} + " on " + {DATASET!r} + ", seed " + {SEED!r}); plt.savefig({png!r}, dpi=120, bbox_inches="tight")
"""
subprocess.run([PY, "-c", script], check=True)
display(Image(png))

## 10. Download the results
In the JupyterLab file browser, go to `localscratch/thesis_v7/novel_continual_learning/experiments/`, right-click a file → **Download**. Or zip the whole folder with the cell below and download `thesis_v7_results.zip` from your home folder. Also keep `environment.txt` (package versions for the thesis), which is included.

In [ ]:
out = shutil.make_archive(os.path.join(HOME, "thesis_v7_results"), "zip", EXPERIMENT_ROOT,
                          "novel_continual_learning/experiments")
with zipfile.ZipFile(out, "a") as z:
    env_txt = os.path.join(EXPERIMENT_ROOT, "environment.txt")
    if os.path.exists(env_txt):
        z.write(env_txt, "environment.txt")
print("Wrote", out)